# Formula 1 Data Cleaning & Preprocessing

This notebook demonstrates the data cleaning and preprocessing pipeline for the Formula 1 historical dataset (sourced from the Ergast API). 

## Objectives:
1. Load the raw datasets (`drivers.csv`, `races.csv`, `results.csv`, `constructors.csv`, `circuits.csv`).
2. Identify and handle missing values (represented as `\N` in the Ergast dataset).
3. Perform type casting (convert columns to appropriate integers, floats, and datetime objects).
4. Feature engineering (create full driver names, positions gained/lost).
5. Export clean datasets for visualization and modeling.

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_DIR = '../Data'
print(f"Data directory: {os.path.abspath(DATA_DIR)}")

## 1. Inspecting the Raw Data
Let's load the drivers dataset and see how missing values are represented.

In [ ]:
drivers_raw = pd.read_csv(os.path.join(DATA_DIR, 'drivers.csv'))
drivers_raw.head(10)

Notice the `\N` in the `number` column (e.g. for Nick Heidfeld, Heikki Kovalainen). In this dataset, `\N` is used to represent missing/NULL values. We need to replace these with actual NaN values so that Pandas can handle them correctly.

In [ ]:
# Replace '\N' with standard NaN
drivers_clean = drivers_raw.replace(r'\\N', np.nan, regex=True)
drivers_clean = drivers_clean.replace(r'\\N', np.nan) # Catch non-string exact matches

# Check for missing values
print("Missing values per column:")
print(drivers_clean.isnull().sum())

## 2. Feature Engineering: Driver Names
Let's combine `forename` and `surname` into a single `driver_name` column, which is much more useful for visualizations and modeling.

In [ ]:
drivers_clean['driver_name'] = drivers_clean['forename'] + ' ' + drivers_clean['surname']
drivers_clean[['driverId', 'driver_name', 'dob', 'nationality']].head()

## 3. Cleaning the Results Dataset
The `results.csv` contains race-by-race finishes. It has many numeric columns that might be read as objects because of the `\N` values. Let's clean and inspect it.

In [ ]:
results_raw = pd.read_csv(os.path.join(DATA_DIR, 'results.csv'))
print("Raw column types:")
print(results_raw.dtypes)

# Replace missing values
results_clean = results_raw.replace(r'\\N', np.nan, regex=True)

# Convert columns to appropriate data types
numeric_cols = ['resultId', 'raceId', 'driverId', 'constructorId', 'grid', 'positionOrder', 'points', 'laps', 'milliseconds', 'rank']
for col in numeric_cols:
    if col in results_clean.columns:
        results_clean[col] = pd.to_numeric(results_clean[col], errors='coerce')

# Cast position (which contains NaN for DNF) to numeric
results_clean['position'] = pd.to_numeric(results_clean['position'], errors='coerce')

print("\nCleaned column types:")
print(results_clean.dtypes)
results_clean.head()

## 4. Merging Datasets
To create a master dataset for easy plotting, we can merge results with races, drivers, and constructors.

In [ ]:
races_raw = pd.read_csv(os.path.join(DATA_DIR, 'races.csv')).replace(r'\\N', np.nan, regex=True)
constructors_raw = pd.read_csv(os.path.join(DATA_DIR, 'constructors.csv')).replace(r'\\N', np.nan, regex=True)

# Rename constructors name for clarity
constructors_clean = constructors_raw.rename(columns={'name': 'constructor_name'})

# Merge
merged_df = results_clean.merge(races_raw[['raceId', 'year', 'round', 'name', 'date']], on='raceId', how='left')
merged_df = merged_df.rename(columns={'name': 'race_name'})

merged_df = merged_df.merge(drivers_clean[['driverId', 'driver_name', 'nationality', 'code']], on='driverId', how='left')
merged_df = merged_df.rename(columns={'nationality': 'driver_nationality'})

merged_df = merged_df.merge(constructors_clean[['constructorId', 'constructor_name', 'nationality']], on='constructorId', how='left')
merged_df = merged_df.rename(columns={'nationality': 'constructor_nationality'})

# Feature engineering: positions gained/lost
merged_df['positions_gained'] = merged_df['grid'] - merged_df['positionOrder']

print(f"Merged shape: {merged_df.shape}")
merged_df[['year', 'race_name', 'driver_name', 'constructor_name', 'grid', 'positionOrder', 'positions_gained', 'points']].head()

## Conclusion
We now have a clean, formatted pipeline for loading and processing raw F1 data in memory. This logic forms the core of our Streamlit dashboard loader.